## Lending Club Predicting Loan Result *Part 3*
### Model training

## Goal
Train a model that can predict whether or not a loan will be paid off on time.

In [ ]:
import pandas as pd
loans = pd.read_csv("clean_loans_2007.csv")
print(loans.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24039 entries, 0 to 24038
Data columns (total 39 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Unnamed: 0                           24039 non-null  int64  
 1   Unnamed: 0.1                         24039 non-null  int64  
 2   loan_amnt                            24039 non-null  float64
 3   int_rate                             24039 non-null  float64
 4   installment                          24039 non-null  float64
 5   emp_length                           24039 non-null  int64  
 6   annual_inc                           24039 non-null  float64
 7   loan_status                          24039 non-null  int64  
 8   dti                                  24039 non-null  float64
 9   delinq_2yrs                          24039 non-null  float64
 10  inq_last_6mths                       24039 non-null  float64
 11  open_acc                    

## Selecting an Error Metric
Since false-positives (loans given out that are never paid back) are so costly (100% of the loan value) compared to the return on paid back loans(~+10% loan value), we should select a metric that minimizes false-positives.

**Optimize** for:
* low [fall-out](https://en.wikipedia.org/wiki/Information_retrieval#Fall-out) (false positive rate)
* high [recall](https://en.wikipedia.org/wiki/Precision_and_recall#Recall) (true positive rate)

**false_positive_rate** = false_positive_cnt / total_negative_total

**true_positive_rate** = true_positive_cnt / total_positive_total

*NOTE:*
* we want to minimize the false_positive_rate (fpr)
* we want to maximize the true_positive_rate  (tpr)

In [ ]:
print(loans.loan_status.value_counts())

1    20405
0     3634
Name: loan_status, dtype: int64


## Training a model on skewed data
With our target column being so heavily biased toward positive predictions (loans payed back), a model will likely overfit to predicting all 1's

In [ ]:
feature_cols = list(loans.columns)
feature_cols.remove('loan_status')
# print(feature_cols)

features = loans[feature_cols]
target = loans['loan_status']

In [ ]:
# testing a simple Logistic Regression model without a train-test split
# metrics true_positive_rate & false_positive_rate
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=3000)
lr.fit(features, target)
predictions = lr.predict(features)

tpr = sum((predictions==1) & (loans.loan_status==1)) / sum(loans.loan_status==1)
fpr = sum((predictions==1) & (loans.loan_status==0)) / sum(loans.loan_status==0)

print("true_positive_rate: ", tpr)
print("false_positive_rate: ", fpr)

true_positive_rate:  0.9970595442293555
false_positive_rate:  0.9942212438084755


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
rf = LogisticRegression(max_iter=3000)

predictions = pd.Series(cross_val_predict(lr, features, target, cv=3))

tpr = sum((predictions==1) & (loans.loan_status==1)) / sum(loans.loan_status==1)
fpr = sum((predictions==1) & (loans.loan_status==0)) / sum(loans.loan_status==0)

print("true_positive_rate: ", tpr)
print("false_positive_rate: ", fpr)

true_positive_rate:  0.8322469982847341
false_positive_rate:  0.7383048981838195


## Correcting for a skewed dataset
### Use oversampling and undersampling
* ensure that the classifier gets input that has a balanced number of each class
* Throw out many rows of data to match the lower skewed amount
* copy the minority classification to match the majority
    *NOTE:* this effectively weights the minority more
* generate fake data. generate variations of minority set

### penalize misclassifications of the less prevalent class more
* weigh the lower skewed category,
* aka set model parameter class_weight='balanced'
* [more info on sklearn's class_weight='balanced'](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn-linear-model-logisticregression)


In [ ]:
# model class_weight parameter to balanced
lr = LogisticRegression(class_weight='balanced', max_iter=3000)
predictions = pd.Series(cross_val_predict(lr, features, target, cv=3))

tpr = sum((predictions==1) & (loans.loan_status==1)) / sum(loans.loan_status==1)
fpr = sum((predictions==1) & (loans.loan_status==0)) / sum(loans.loan_status==0)

print("true_positive_rate: ", tpr)
print("false_positive_rate: ", fpr)

true_positive_rate:  0.6273462386669933
false_positive_rate:  0.47578425976884975


## adjusting classification penalties to match their monetary reward/loss
The disparity in profit for a models loan prediction is due to the large(100%) monetary loss of false positives vs the gain of successful loans(10%).
This should drive the model's weights more than the skewed category of defaulted loans.

In [ ]:
penalty = {
    0: 10,
    1: 1
}
lr = LogisticRegression(class_weight=penalty, max_iter=3000)
predictions = pd.Series(cross_val_predict(lr, features, target, cv=3))

tpr = sum((predictions==1) & (loans.loan_status==1)) / sum(loans.loan_status==1)
fpr = sum((predictions==1) & (loans.loan_status==0)) / sum(loans.loan_status==0)

print("true_positive_rate: ", tpr)
print("false_positive_rate: ", fpr)

true_positive_rate:  0.5052683165890713
false_positive_rate:  0.36818932305998897


In [ ]:
# try training a model on a problem specific penalty dict
from sklearn.ensemble import RandomForestClassifier
penalty = {
    0: 10,
    1: 1
}
lr = RandomForestClassifier(class_weight=penalty, random_state=1)
predictions = pd.Series(cross_val_predict(lr, features, target, cv=3))

tpr = sum((predictions==1) & (loans.loan_status==1)) / sum(loans.loan_status==1)
fpr = sum((predictions==1) & (loans.loan_status==0)) / sum(loans.loan_status==0)

print("true_positive_rate: ", tpr)
print("false_positive_rate: ", fpr)

true_positive_rate:  0.6295515804949767
false_positive_rate:  0.6075949367088608


## Ideas for improvement
* tweak the penalties further.
* try models other than a random forest and logistic regression.
* use some of the columns we discarded to generate better features.
* ensemble multiple models to get more accurate predictions.
* tune the parameters of the algorithm to achieve higher performance.